In [ ]:
# ===== CONFIG =====
import os

INPUT_DIR   = os.environ.get("ITDA_INPUT_DIR",  "./val_images")
OUTPUT_PATH = os.environ.get("ITDA_OUTPUT_PATH", "./submission.csv")
# ==================


In [ ]:
USE_GPU = False

import os
import re
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"

import torch
import torchvision
import doctr

import paddle

from PIL import Image, ImageOps, ImageFilter
from paddleocr import PaddleOCR

if USE_GPU:
    raise RuntimeError("USE_GPU must be False.")

DEVICE = "cpu"
paddle.set_device(DEVICE)

IMAGE_DIR = Path(INPUT_DIR)
RESOLUTION = 1280

MODEL_ROOT = Path("./models")
DET_MODEL_DIR = MODEL_ROOT / "PP-OCRv5_mobile_det"
REC_MODEL_DIR = MODEL_ROOT / "korean_PP-OCRv5_mobile_rec"
DOC_ORI_MODEL_DIR = MODEL_ROOT / "PP-LCNet_x1_0_doc_ori"
TEXTLINE_ORI_MODEL_DIR = MODEL_ROOT / "PP-LCNet_x1_0_textline_ori"

required_paths = [
    DET_MODEL_DIR,
    REC_MODEL_DIR,
    DOC_ORI_MODEL_DIR,
    TEXTLINE_ORI_MODEL_DIR,
]

missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Required offline PaddleOCR model files are missing:\n" + "\n".join(missing_paths)
    )

ocr = PaddleOCR(
    text_detection_model_name="PP-OCRv5_mobile_det",
    text_detection_model_dir=str(DET_MODEL_DIR),
    text_recognition_model_name="korean_PP-OCRv5_mobile_rec",
    text_recognition_model_dir=str(REC_MODEL_DIR),
    device=DEVICE,
    cpu_threads=4,
    enable_mkldnn=True,
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    text_rec_score_thresh=0.0,
)

orientation_ocr = None

def get_orientation_ocr():
    global orientation_ocr

    if orientation_ocr is None:
        orientation_ocr = PaddleOCR(
            text_detection_model_name="PP-OCRv5_mobile_det",
            text_detection_model_dir=str(DET_MODEL_DIR),
            text_recognition_model_name="korean_PP-OCRv5_mobile_rec",
            text_recognition_model_dir=str(REC_MODEL_DIR),
            doc_orientation_classify_model_name="PP-LCNet_x1_0_doc_ori",
            doc_orientation_classify_model_dir=str(DOC_ORI_MODEL_DIR),
            textline_orientation_model_name="PP-LCNet_x1_0_textline_ori",
            textline_orientation_model_dir=str(TEXTLINE_ORI_MODEL_DIR),
            device=DEVICE,
            cpu_threads=4,
            enable_mkldnn=True,
            use_doc_orientation_classify=True,
            use_doc_unwarping=False,
            use_textline_orientation=True,
            text_rec_score_thresh=0.0,
        )

    return orientation_ocr

def prepare_image(
    path,
    resolution=1280
):

    with Image.open(path) as source:

        image = ImageOps.exif_transpose(
            source
        ).convert(
            "RGB"
        )

        image.thumbnail(
            (
                resolution,
                resolution
            ),
            Image.Resampling.LANCZOS
        )

        image = ImageOps.autocontrast(
            image,
            cutoff=1
        )

        return image

def adaptive_threshold_image(
    image,
    radius=12,
    offset=8,
    invert=False
):

    gray = ImageOps.grayscale(
        image
    )

    gray = ImageOps.autocontrast(
        gray,
        cutoff=1
    )

    arr = np.asarray(
        gray,
        dtype=np.float32
    )

    local_mean_img = gray.filter(
        ImageFilter.BoxBlur(
            radius
        )
    )

    local_mean = np.asarray(
        local_mean_img,
        dtype=np.float32
    )

    if invert:

        binary = np.where(
            arr > local_mean + offset,
            255,
            0
        )

    else:

        binary = np.where(
            arr < local_mean - offset,
            0,
            255
        )

    binary = binary.astype(
        np.uint8
    )

    rgb = np.stack(
        [
            binary,
            binary,
            binary
        ],
        axis=-1
    )

    return Image.fromarray(
        rgb
    )

def run_ocr(image, orientation=False):
    bgr = np.ascontiguousarray(np.asarray(image)[:, :, ::-1])

    if orientation:
        engine = get_orientation_ocr()
        result = engine.predict(
            bgr,
            use_doc_orientation_classify=True,
            use_doc_unwarping=False,
            use_textline_orientation=True
        )
    else:
        result = ocr.predict(
            bgr,
            use_doc_orientation_classify=False,
            use_doc_unwarping=False,
            use_textline_orientation=False
        )

    if not result:
        return [], []

    res = result[0]

    try:
        texts = res["rec_texts"] if "rec_texts" in res else []
        scores = res["rec_scores"] if "rec_scores" in res else []
    except Exception:
        texts = []
        scores = []

    return texts, scores

def make_valid_date(
    year,
    month,
    day
):

    try:

        year = int(year)
        month = int(month)
        day = int(day)

        if (
            year < 2018
            or
            year > 2032
        ):

            return None

        d = datetime(
            year,
            month,
            day
        )

        return d.strftime(
            "%Y-%m-%d"
        )

    except Exception:

        return None

NUMERIC_CONFUSIONS = {

    "O": "0",
    "o": "0",
    "U": "0",

    "I": "1",
    "l": "1",
    "|": "1",

    "Z": "2",
    "z": "2",

    "S": "5",
    "s": "5",

    "B": "8",
}

def normalize_text(
    text
):

    text = str(
        text
    )

    replacements = {

        "：": ":",
        "，": ",",
        "．": ".",
        "／": "/",
        "－": "-",
    }

    for old, new in replacements.items():

        text = text.replace(
            old,
            new
        )

    chars = list(
        text
    )

    for i, char in enumerate(
        chars
    ):

        if char not in NUMERIC_CONFUSIONS:

            continue

        prev_char = (
            chars[i - 1]
            if i > 0
            else ""
        )

        next_char = (
            chars[i + 1]
            if i + 1 < len(chars)
            else ""
        )

        prev_ok = (
            prev_char.isdigit()
            or
            prev_char in "./-:,"
        )

        next_ok = (
            next_char.isdigit()
            or
            next_char in "./-:,"
        )

        if (
            prev_ok
            and
            next_ok
        ):

            chars[i] = (
                NUMERIC_CONFUSIONS[
                    char
                ]
            )

    return "".join(
        chars
    )

def detect_date_order_hint(
    texts
):

    full_text = " ".join(
        str(x)
        for x in texts
    ).upper()

    dmy_hints = [

        "일/월/년",
        "일-월-년",

        "DD/MM/YYYY",
        "DD-MM-YYYY",
        "DD/MM/YY",

        "DAY/MONTH/YEAR",
    ]

    mdy_hints = [

        "월/일/년",
        "월-일-년",

        "MM/DD/YYYY",
        "MM-DD-YYYY",
        "MM/DD/YY",

        "MONTH/DAY/YEAR",
    ]

    ymd_hints = [

        "년/월/일",
        "년-월-일",

        "YYYY/MM/DD",
        "YYYY-MM-DD",

        "YEAR/MONTH/DAY",
    ]

    for hint in dmy_hints:

        if hint in full_text:

            return "DMY"

    for hint in mdy_hints:

        if hint in full_text:

            return "MDY"

    for hint in ymd_hints:

        if hint in full_text:

            return "YMD"

    return None

def add_candidate(
    result,
    raw,
    date,
    fmt,
    fuzzy=False
):

    if date is None:

        return

    result.append({

        "raw":
            raw,

        "date":
            date,

        "format":
            fmt,

        "fuzzy":
            fuzzy,
    })

def repair_three_digit_year(
    year3
):

    year3 = str(
        year3
    )

    if (
        len(year3) == 3
        and
        year3.startswith("2")
    ):

        return int(
            "20"
            +
            year3[1:]
        )

    return None

def extract_date_candidates(
    text
):

    text = normalize_text(
        text
    )

    candidates = []

    pattern = re.compile(

        r"(?<!\d)"

        r"(20\d{2})"

        r"\s*[-./:,년\s]\s*"

        r"(\d{1,2})"

        r"\s*[-./:,월\s]\s*"

        r"(\d{1,2})"

        r"\s*일?"
    )

    for m in pattern.finditer(
        text
    ):

        y, mo, d = m.groups()

        add_candidate(

            candidates,

            m.group(0),

            make_valid_date(
                y,
                mo,
                d
            ),

            "YMD"
        )

    pattern = re.compile(

        r"(?<!\d)"

        r"(20\d{2})"

        r"(\d{2})"

        r"(\d{2})"

        r"(?!\d)"
    )

    for m in pattern.finditer(
        text
    ):

        y, mo, d = m.groups()

        add_candidate(

            candidates,

            m.group(0),

            make_valid_date(
                y,
                mo,
                d
            ),

            "YMD"
        )

    pattern = re.compile(

        r"(?<!\d)"

        r"(20\d{2})"

        r"\s*[-./:,]\s*"

        r"(\d{2})"

        r"(\d{2})"

        r"(?!\d)"
    )

    for m in pattern.finditer(
        text
    ):

        y, mo, d = m.groups()

        add_candidate(

            candidates,

            m.group(0),

            make_valid_date(
                y,
                mo,
                d
            ),

            "YMD"
        )

    pattern = re.compile(

        r"(?<!\d)"

        r"(\d{2})"

        r"\s*[-./:,]\s*"

        r"(\d{1,2})"

        r"\s*[-./:,]\s*"

        r"(\d{1,2})"

        r"(?!\d)"
    )

    for m in pattern.finditer(
        text
    ):

        yy, mo, d = m.groups()

        y = (
            2000
            +
            int(yy)
        )

        add_candidate(

            candidates,

            m.group(0),

            make_valid_date(
                y,
                mo,
                d
            ),

            "YMD"
        )

    pattern = re.compile(

        r"(?<!\d)"

        r"(\d{2})"

        r"(\d{2})"

        r"(\d{2})"

        r"(?!\d)"
    )

    for m in pattern.finditer(
        text
    ):

        yy, mo, d = m.groups()

        y = (
            2000
            +
            int(yy)
        )

        add_candidate(

            candidates,

            m.group(0),

            make_valid_date(
                y,
                mo,
                d
            ),

            "YMD"
        )

    pattern = re.compile(

        r"(?<!\d)"

        r"(\d{1,2})"

        r"\s*[-./:,]\s*"

        r"(\d{1,2})"

        r"\s*[-./:,]\s*"

        r"(20\d{2})"

        r"(?!\d)"
    )

    for m in pattern.finditer(
        text
    ):

        first, second, y = (
            m.groups()
        )

        dmy = make_valid_date(
            y,
            second,
            first
        )

        mdy = make_valid_date(
            y,
            first,
            second
        )

        add_candidate(

            candidates,

            m.group(0),

            dmy,

            "DMY"
        )

        if mdy != dmy:

            add_candidate(

                candidates,

                m.group(0),

                mdy,

                "MDY"
            )

    pattern = re.compile(

        r"(?<!\d)"

        r"(2\d{2})"

        r"\s*[-./:,]\s*"

        r"(\d{1,2})"

        r"\s*[-./:,]\s*"

        r"(\d{1,2})"

        r"(?!\d)"
    )

    for m in pattern.finditer(
        text
    ):

        year3, mo, d = (
            m.groups()
        )

        y = repair_three_digit_year(
            year3
        )

        add_candidate(

            candidates,

            m.group(0),

            (
                make_valid_date(
                    y,
                    mo,
                    d
                )
                if y
                else None
            ),

            "YMD_3YEAR",

            True
        )

    unique = []

    seen = set()

    for c in candidates:

        key = (

            c["date"],
            c["raw"],
            c["format"],
        )

        if key not in seen:

            seen.add(
                key
            )

            unique.append(
                c
            )

    return unique

def build_stage_candidates(
    texts,
    scores,
    stage
):

    result = []

    hint = detect_date_order_hint(
        texts
    )

    for i, text in enumerate(
        texts
    ):

        parsed = (
            extract_date_candidates(
                text
            )
        )

        if not parsed:

            continue

        confidence = (

            float(scores[i])

            if i < len(scores)

            else 0.0
        )

        for candidate in parsed:

            hint_match = (

                hint is not None

                and

                candidate["format"]
                ==
                hint
            )

            result.append({

                "date":
                    candidate["date"],

                "raw":
                    candidate["raw"],

                "format":
                    candidate["format"],

                "fuzzy":
                    candidate["fuzzy"],

                "confidence":
                    confidence,

                "stage":
                    stage,

                "hint_match":
                    hint_match,
            })

    return result

def aggregate_candidates(
    candidates
):

    grouped = {}

    for c in candidates:

        date = c["date"]

        if date not in grouped:

            grouped[date] = []

        grouped[date].append(
            c
        )

    result = []

    for date, items in grouped.items():

        best_item = max(

            items,

            key=lambda x:
                x["confidence"]
        )

        stages = set(

            x["stage"]

            for x in items
        )

        hint_match = any(

            x["hint_match"]

            for x in items
        )

        result.append({

            **best_item,

            "agreement_count":
                len(stages),

            "support_stages":
                ",".join(
                    sorted(stages)
                ),

            "hint_match_any":
                hint_match,
        })

    return result

def select_latest_candidate(
    aggregated
):

    if not aggregated:

        return None

    hinted = [

        c

        for c in aggregated

        if c["hint_match_any"]
    ]

    if hinted:

        pool = hinted

    else:

        pool = aggregated

    return max(

        pool,

        key=lambda c:
            datetime.strptime(
                c["date"],
                "%Y-%m-%d"
            )
    )

def get_low_confidence_reason(
    best
):

    if best is None:

        return "NO_CANDIDATE"

    raw = str(

        best.get(
            "raw",
            ""
        )

    ).strip()

    confidence = float(

        best.get(
            "confidence",
            0
        )
    )

    agreement_count = int(

        best.get(
            "agreement_count",
            1
        )
    )

    reasons = []

    if (

        best.get(
            "fuzzy",
            False
        )

        and

        agreement_count == 1
    ):

        reasons.append(
            "FUZZY"
        )

    if (

        confidence < 0.90

        and

        agreement_count == 1
    ):

        reasons.append(
            "LOW_CONF"
        )

    if re.fullmatch(
        r"\d{6}",
        raw
    ):

        reasons.append(
            "PURE_6DIGIT"
        )

    if re.fullmatch(
        r"\d{8}",
        raw
    ):

        reasons.append(
            "PURE_8DIGIT"
        )

    standard_parts = re.fullmatch(

        r"(\d{2,4})"

        r"([.\-/:,])"

        r"(\d{1,2})"

        r"\2"

        r"(\d{1,2})",

        raw
    )

    if standard_parts:

        month_part = (
            standard_parts.group(3)
        )

        day_part = (
            standard_parts.group(4)
        )

        if (

            len(month_part) == 1

            or

            len(day_part) == 1
        ):

            reasons.append(
                "SINGLE_DIGIT_MD"
            )

    if re.fullmatch(
        r"\d{2}-\d{2}-\d{2}",
        raw
    ):

        reasons.append(
            "YY_HYPHEN"
        )

    if re.fullmatch(
        r"\d{4}\.\d{2}\s+\d{2}",
        raw
    ):

        reasons.append(
            "MIXED_SEPARATOR"
        )

    if reasons:

        return "|".join(
            reasons
        )

    return ""

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
}

image_paths = [

    path

    for path in IMAGE_DIR.iterdir()

    if (
        path.is_file()

        and

        path.suffix.lower()
        in IMAGE_EXTENSIONS
    )
]

def image_sort_key(
    path
):

    stem = path.stem

    if stem.isdigit():

        return (
            0,
            int(stem)
        )

    return (
        1,
        stem
    )

image_paths = sorted(
    image_paths,
    key=image_sort_key
)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

image_paths = sorted(
    [
        path for path in IMAGE_DIR.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    ],
    key=lambda path: (0, int(path.stem)) if path.stem.isdigit() else (1, path.stem)
)

prediction_rows = []
low_confidence_paddle = []

for image_path in image_paths:
    image_id = image_path.stem

    try:
        image = prepare_image(image_path, RESOLUTION)
        all_candidates = []

        texts, scores = run_ocr(image, orientation=False)
        all_candidates.extend(build_stage_candidates(texts, scores, "BASE"))

        if len(all_candidates) == 0:
            adaptive = adaptive_threshold_image(image, invert=False)
            texts, scores = run_ocr(adaptive, orientation=False)
            all_candidates.extend(build_stage_candidates(texts, scores, "ADAPTIVE_DARK"))

        if len(all_candidates) == 0:
            adaptive = adaptive_threshold_image(image, invert=True)
            texts, scores = run_ocr(adaptive, orientation=False)
            all_candidates.extend(build_stage_candidates(texts, scores, "ADAPTIVE_LIGHT"))

        if len(all_candidates) == 0:
            texts, scores = run_ocr(image, orientation=True)
            all_candidates.extend(build_stage_candidates(texts, scores, "ORIENTATION"))

        aggregated = aggregate_candidates(all_candidates)
        best = select_latest_candidate(aggregated)

        if best is None:
            predicted_date = None
            low_confidence = True
        else:
            predicted_date = best["date"]
            low_confidence = get_low_confidence_reason(best) != ""

        if low_confidence:
            low_confidence_paddle.append(image_id)

        if predicted_date is not None:
            dt = datetime.strptime(predicted_date, "%Y-%m-%d")
            year = dt.strftime("%Y")
            month = dt.strftime("%m")
            day = dt.strftime("%d")
            final_date = predicted_date
        else:
            year = None
            month = None
            day = None
            final_date = None

        prediction_rows.append({
            "image_id": image_id,
            "year": year,
            "month": month,
            "day": day,
            "final_date": final_date,
        })

    except Exception:
        low_confidence_paddle.append(image_id)
        prediction_rows.append({
            "image_id": image_id,
            "year": None,
            "month": None,
            "day": None,
            "final_date": None,
        })

paddle_predictions = pd.DataFrame(prediction_rows)


In [ ]:
DOCTR_IMAGE_DIR = Path(INPUT_DIR).expanduser()
DOCTR_WEIGHTS_DIR = MODEL_ROOT / "doctr"
DOCTR_RESOLUTION = 1280

def doctr_normalize_image_id(value):
    value = str(value).strip()
    if not value or "/" in value or "\\" in value:
        raise ValueError(f"잘못된 이미지 ID: {value!r}")
    return str(int(value)) if value.isdecimal() else value

DOCTR_IMAGE_INDEX = {}

for path in image_paths:
    key = doctr_normalize_image_id(path.stem)

    if key in DOCTR_IMAGE_INDEX:
        raise ValueError(
            f"중복 이미지 ID: {DOCTR_IMAGE_INDEX[key]} / {path}"
        )

    DOCTR_IMAGE_INDEX[key] = path


def doctr_find_image(image_id):
    return DOCTR_IMAGE_INDEX.get(
        doctr_normalize_image_id(image_id)
    )

DOCTR_PADDLE_LOW_IDS = []
seen_ids = set()

for value in low_confidence_paddle:
    key = doctr_normalize_image_id(value)
    if key not in seen_ids:
        seen_ids.add(key)
        DOCTR_PADDLE_LOW_IDS.append(str(value))

doctr_model = None
doctr_available = False

try:
    import torch
    from doctr.models import ocr_predictor

    torch.set_num_threads(4)

    det_path = DOCTR_WEIGHTS_DIR / "detector.pt"
    reco_path = DOCTR_WEIGHTS_DIR / "recognizer.pt"

    if not det_path.is_file() or not reco_path.is_file():
        raise FileNotFoundError("Offline docTR weights are missing.")

    doctr_model = ocr_predictor(
        det_arch="db_mobilenet_v3_large",
        reco_arch="crnn_mobilenet_v3_small",
        pretrained=False,
        pretrained_backbone=False,
        assume_straight_pages=True,
        preserve_aspect_ratio=True,
    )

    doctr_model.det_predictor.model.load_state_dict(
        torch.load(det_path, map_location="cpu", weights_only=True),
        strict=True,
    )

    doctr_model.reco_predictor.model.load_state_dict(
        torch.load(reco_path, map_location="cpu", weights_only=True),
        strict=True,
    )

    doctr_model = doctr_model.cpu().eval()
    doctr_available = True

except (ImportError, OSError, RuntimeError, FileNotFoundError):
    doctr_model = None
    doctr_available = False


def doctr_prepare_image(path, resolution=1280):
    with Image.open(path) as source:
        image = ImageOps.exif_transpose(
            source
        ).convert("RGB")

        image.thumbnail(
            (resolution, resolution),
            Image.Resampling.LANCZOS
        )

        image = ImageOps.autocontrast(
            image,
            cutoff=1
        )

        return image

def doctr_run_doctr(image):
    rgb = np.ascontiguousarray(np.asarray(image))

    with torch.inference_mode():
        result = doctr_model([rgb])

    texts = []
    scores = []

    for page in result.pages:
        for block in page.blocks:
            for line in block.lines:
                for word in line.words:
                    texts.append(str(word.value))
                    scores.append(float(word.confidence))

    return texts, scores

def doctr_make_valid_date(year, month, day):
    try:
        year = int(year)
        month = int(month)
        day = int(day)

        if not (2018 <= year <= 2032):
            return None

        dt = datetime(
            year,
            month,
            day
        )

        return dt.strftime(
            "%Y-%m-%d"
        )

    except (TypeError, ValueError):
        return None

def doctr_normalize_text(text):
    text = str(text)

    replacements = {
        "：": ":",
        "，": ",",
        "．": ".",
        "／": "/",
        "－": "-",
        "｜": "|",
        "–": "-",
        "—": "-"
    }

    for old, new in replacements.items():
        text = text.replace(
            old,
            new
        )

    return text

DOCTR_POSITIVE_KEYWORDS = [
    "소비기한",
    "유통기한",
    "사용기한",
    "품질유지기한",
    "EXP",
    "EXP.",
    "EXPIRY",
    "EXPIRE",
    "EXPIRES",
    "EXPIRATION",
    "BEST BEFORE",
    "BEST BY",
    "USE BY",
    "USE-BY",
    "SELL BY",
    "BBE",
    "까지",
]

DOCTR_NEGATIVE_KEYWORDS = [
    "제조일",
    "제조일자",
    "제조",
    "MFG",
    "MANUFACTURED",
]

def doctr_context_score(text, start, end):
    context = text[
        max(0, start - 35):
        min(len(text), end + 35)
    ].upper()

    context = re.sub(
        r"\s*\|\s*",
        " ",
        context
    )

    context = re.sub(
        r"\s+",
        " ",
        context
    )

    score = 0

    for keyword in DOCTR_POSITIVE_KEYWORDS:
        if keyword.upper() in context:
            score += 2

    for keyword in DOCTR_NEGATIVE_KEYWORDS:
        if keyword.upper() in context:
            score -= 2

    return score

def doctr_join_words_with_spans(texts):
    pieces = []
    spans = []
    cursor = 0

    for i, text in enumerate(texts):
        text = str(text)

        if i > 0:
            separator = " | "
            pieces.append(separator)
            cursor += len(separator)

        start = cursor
        pieces.append(text)
        cursor += len(text)
        end = cursor

        spans.append(
            (start, end)
        )

    return "".join(pieces), spans

def doctr_candidate_confidence(
    start,
    end,
    spans,
    scores
):
    matched_scores = []

    for i, (word_start, word_end) in enumerate(spans):
        overlap = (
            word_end > start
            and word_start < end
        )

        if overlap and i < len(scores):
            matched_scores.append(
                float(scores[i])
            )

    if not matched_scores:
        return 0.0

    return float(
        np.mean(matched_scores)
    )

def doctr_append_candidate(
    candidates,
    match,
    date,
    fuzzy=False
):
    if date is None:
        return

    candidates.append({
        "date": date,
        "raw": match.group(0),
        "start": match.start(),
        "end": match.end(),
        "fuzzy": fuzzy
    })

def doctr_extract_date_candidates(text):
    text = doctr_normalize_text(text)
    candidates = []

    pattern = re.compile(
        r"(?<!\d)"
        r"(20\d{2})"
        r"\s*[.\-/,:|년월일\s]{1,8}\s*"
        r"(\d{1,2})"
        r"\s*[.\-/,:|년월일\s]{1,8}\s*"
        r"(\d{1,2})"
        r"(?!\d)"
    )

    for m in pattern.finditer(text):
        y, mo, d = m.groups()

        doctr_append_candidate(
            candidates,
            m,
            doctr_make_valid_date(
                y, mo, d
            )
        )

    pattern = re.compile(
        r"(?<!\d)"
        r"(20\d{2})"
        r"(0[1-9]|1[0-2])"
        r"(0[1-9]|[12]\d|3[01])"
        r"(?!\d)"
    )

    for m in pattern.finditer(text):
        y, mo, d = m.groups()

        doctr_append_candidate(
            candidates,
            m,
            doctr_make_valid_date(
                y, mo, d
            )
        )

    pattern = re.compile(
        r"(?<!\d)"
        r"(20\d{2})"
        r"(0[1-9]|1[0-2])"
        r"\s*[.\-/,:]\s*"
        r"(0[1-9]|[12]\d|3[01])"
        r"(?!\d)"
    )

    for m in pattern.finditer(text):
        y, mo, d = m.groups()

        doctr_append_candidate(
            candidates,
            m,
            doctr_make_valid_date(
                y, mo, d
            )
        )

    pattern = re.compile(
        r"(?<!\d)"
        r"(20\d{2})"
        r"\s*[.\-/,:]\s*"
        r"(0[1-9]|1[0-2])"
        r"(0[1-9]|[12]\d|3[01])"
        r"(?!\d)"
    )

    for m in pattern.finditer(text):
        y, mo, d = m.groups()

        doctr_append_candidate(
            candidates,
            m,
            doctr_make_valid_date(
                y, mo, d
            )
        )

    pattern = re.compile(
        r"(?<!\d)"
        r"(2\d)"
        r"\s*[.\-/,:|\s]{1,8}\s*"
        r"(\d{1,2})"
        r"\s*[.\-/,:|\s]{1,8}\s*"
        r"(\d{1,2})"
        r"(?!\d)"
    )

    for m in pattern.finditer(text):
        yy, mo, d = m.groups()
        y = 2000 + int(yy)

        doctr_append_candidate(
            candidates,
            m,
            doctr_make_valid_date(
                y, mo, d
            )
        )

    pattern = re.compile(
        r"(?<!\d)"
        r"(2\d)"
        r"(0[1-9]|1[0-2])"
        r"(0[1-9]|[12]\d|3[01])"
        r"(?!\d)"
    )

    for m in pattern.finditer(text):
        yy, mo, d = m.groups()
        y = 2000 + int(yy)

        doctr_append_candidate(
            candidates,
            m,
            doctr_make_valid_date(
                y, mo, d
            )
        )

    pattern = re.compile(
        r"(?<!\d)"
        r"(\d{1,2})"
        r"\s*[.\-/,:|\s]{1,8}\s*"
        r"(\d{1,2})"
        r"\s*[.\-/,:|\s]{1,8}\s*"
        r"(20\d{2})"
        r"(?!\d)"
    )

    for m in pattern.finditer(text):
        first, second, year = m.groups()

        dmy_date = doctr_make_valid_date(
            year,
            second,
            first
        )

        if dmy_date is not None:
            doctr_append_candidate(
                candidates,
                m,
                dmy_date
            )

        mdy_date = doctr_make_valid_date(
            year,
            first,
            second
        )

        if (
            mdy_date is not None
            and mdy_date != dmy_date
        ):
            doctr_append_candidate(
                candidates,
                m,
                mdy_date
            )

    for candidate in candidates:
        candidate["context_score"] = (
            doctr_context_score(
                text,
                candidate["start"],
                candidate["end"]
            )
        )

    unique = []
    seen = set()

    for candidate in candidates:
        key = (
            candidate["date"],
            candidate["raw"]
        )

        if key not in seen:
            seen.add(key)
            unique.append(
                candidate
            )

    return unique

def doctr_build_stage_candidates(
    texts,
    scores,
    stage="BASE"
):
    if not texts:
        return []

    full_text, spans = (
        doctr_join_words_with_spans(
            texts
        )
    )

    parsed = doctr_extract_date_candidates(
        full_text
    )

    result = []

    for candidate in parsed:
        confidence = doctr_candidate_confidence(
            candidate["start"],
            candidate["end"],
            spans,
            scores
        )

        result.append({
            "date": candidate["date"],
            "raw": candidate["raw"],
            "fuzzy": candidate["fuzzy"],
            "confidence": confidence,
            "context_score": candidate.get(
                "context_score",
                0
            ),
            "start": candidate["start"],
            "stage": stage
        })

    return result

def doctr_aggregate_candidates(candidates):
    grouped = {}

    for candidate in candidates:
        date = candidate["date"]

        grouped.setdefault(
            date,
            []
        ).append(
            candidate
        )

    result = []

    for date, items in grouped.items():
        best_item = max(
            items,
            key=lambda x: (
                x.get("context_score", 0),
                x.get("confidence", 0)
            )
        )

        best_context_score = max(
            x.get("context_score", 0)
            for x in items
        )

        earliest_start = min(
            x.get("start", 0)
            for x in items
        )

        result.append({
            **best_item,
            "context_score": best_context_score,
            "start": earliest_start
        })

    return result

def doctr_select_best_candidate(aggregated):
    if not aggregated:
        return None

    return max(
        aggregated,
        key=lambda candidate: (
            candidate.get(
                "context_score",
                0
            ),
            candidate.get(
                "confidence",
                0
            ),
            -candidate.get(
                "start",
                0
            )
        )
    )

def doctr_is_doctr_low_confidence(
    best,
    aggregated
):
    if best is None:
        return True

    if best.get(
        "context_score",
        0
    ) < 0:
        return True

    if len(aggregated) == 1:
        return False

    context_scores = [
        c.get(
            "context_score",
            0
        )
        for c in aggregated
    ]

    max_context = max(
        context_scores
    )

    top_count = sum(
        score == max_context
        for score in context_scores
    )

    if (
        max_context > 0
        and top_count == 1
        and best.get(
            "context_score",
            0
        ) == max_context
    ):
        return False

    return True

doctr_results = {}
low_confidence_docTR = []

if doctr_available:
    for image_id in DOCTR_PADDLE_LOW_IDS:
        image_path = doctr_find_image(image_id)

        if image_path is None:
            low_confidence_docTR.append(str(image_id))
            continue

        image_id = image_path.stem

        try:
            image = doctr_prepare_image(image_path, DOCTR_RESOLUTION)
            texts, scores = doctr_run_doctr(image)
            candidates = doctr_build_stage_candidates(texts, scores, "BASE")
            aggregated = doctr_aggregate_candidates(candidates)
            best = doctr_select_best_candidate(aggregated)
            doctr_low = doctr_is_doctr_low_confidence(best, aggregated)

            if doctr_low:
                low_confidence_docTR.append(image_id)
            else:
                doctr_results[image_id] = best["date"]

        except Exception:
            low_confidence_docTR.append(image_id)
else:
    low_confidence_docTR = list(DOCTR_PADDLE_LOW_IDS)

final_results = paddle_predictions.copy()

for image_id, new_date in doctr_results.items():
    mask = final_results["image_id"].astype(str) == str(image_id)

    if mask.any():
        dt = datetime.strptime(new_date, "%Y-%m-%d")
        final_results.loc[mask, "year"] = dt.strftime("%Y")
        final_results.loc[mask, "month"] = dt.strftime("%m")
        final_results.loc[mask, "day"] = dt.strftime("%d")
        final_results.loc[mask, "final_date"] = new_date


In [ ]:

from rapidocr_onnxruntime import RapidOCR

RAPID_WEIGHTS_DIR = MODEL_ROOT / "rapidocr"

RAPIDOCR_RESOLUTION = 840

def rapid_normalize_text(text):
    replacements = {
        "：": ":",
        "，": ",",
        "．": ".",
        "／": "/",
        "－": "-",
        "–": "-",
        "—": "-",
        "년": "-",
        "월": "-",
        "일": "",
    }

    text = str(text)

    for old, new in replacements.items():
        text = text.replace(old, new)

    return text


def rapid_valid_full_date(year, month, day):
    try:
        year = int(year)
        month = int(month)
        day = int(day)

        if not 2018 <= year <= 2032:
            return None

        datetime(year, month, day)

        return {
            "year": f"{year:04d}",
            "month": f"{month:02d}",
            "day": f"{day:02d}",
            "final_date": f"{year:04d}-{month:02d}-{day:02d}",
        }

    except (TypeError, ValueError):
        return None


def rapid_valid_month_day(month, day):
    try:
        month = int(month)
        day = int(day)

        datetime(2024, month, day)

        return {
            "year": "NONE",
            "month": f"{month:02d}",
            "day": f"{day:02d}",
            "final_date": f"NONE-{month:02d}-{day:02d}",
        }

    except (TypeError, ValueError):
        return None


RAPID_POSITIVE_KEYWORDS = [
    "소비기한",
    "유통기한",
    "사용기한",
    "품질유지기한",
    "EXP",
    "EXPIRY",
    "EXPIRE",
    "EXPIRES",
    "EXPIRATION",
    "BEST BEFORE",
    "BEST BY",
    "USE BY",
    "USE-BY",
    "SELL BY",
    "BBE",
    "까지",
]

RAPID_NEGATIVE_KEYWORDS = [
    "제조일",
    "제조일자",
    "제조",
    "MFG",
    "MANUFACTURED",
]


def rapid_context_score(text, start, end):
    context = text[
        max(0, start - 35):
        min(len(text), end + 35)
    ].upper()

    return (
        sum(
            2
            for keyword in RAPID_POSITIVE_KEYWORDS
            if keyword.upper() in context
        )
        -
        sum(
            2
            for keyword in RAPID_NEGATIVE_KEYWORDS
            if keyword.upper() in context
        )
    )


def rapid_extract_date_candidates(text):
    if not isinstance(text, str) or not text.strip():
        return []

    normalized = rapid_normalize_text(text)
    candidates = []
    full_date_spans = []

    def add_candidate(info, match, kind):
        if info:
            candidates.append({
                **info,
                "raw": match.group(0),
                "start": match.start(),
                "end": match.end(),
                "kind": kind,
                "context_score": rapid_context_score(
                    normalized,
                    match.start(),
                    match.end()
                ),
            })

    patterns = [
        (
            r"(?<!\d)(20\d{2})\s*[-./,:|\s]+\s*(\d{1,2})\s*[-./,:|\s]+\s*(\d{1,2})(?!\d)",
            "YMD",
        ),
        (
            r"(?<!\d)(20\d{2})(\d{2})(\d{2})(?!\d)",
            "YMD_COMPACT",
        ),
        (
            r"(?<!\d)(\d{2})\s*[-./,:|\s]+\s*(\d{1,2})\s*[-./,:|\s]+\s*(\d{1,2})(?!\d)",
            "YMD_2YEAR",
        ),
    ]

    for pattern_text, kind in patterns:
        for match in re.finditer(pattern_text, normalized):
            groups = match.groups()

            if kind == "YMD_2YEAR":
                info = rapid_valid_full_date(
                    2000 + int(groups[0]),
                    groups[1],
                    groups[2],
                )
            else:
                info = rapid_valid_full_date(*groups)

            if info:
                add_candidate(info, match, kind)
                full_date_spans.append(
                    (match.start(), match.end())
                )

    pattern = (
        r"(?<!\d)"
        r"(\d{1,2})"
        r"\s*[-./,:|\s]+\s*"
        r"(\d{1,2})"
        r"\s*[-./,:|\s]+\s*"
        r"(20\d{2})"
        r"(?!\d)"
    )

    for match in re.finditer(pattern, normalized):
        first, second, year = match.groups()

        dmy = rapid_valid_full_date(
            year,
            second,
            first
        )

        mdy = rapid_valid_full_date(
            year,
            first,
            second
        )

        if dmy:
            add_candidate(dmy, match, "DMY")

        if mdy and (
            not dmy
            or mdy["final_date"] != dmy["final_date"]
        ):
            add_candidate(mdy, match, "MDY")

        if dmy or mdy:
            full_date_spans.append(
                (match.start(), match.end())
            )

    for match in re.finditer(
        r"(?<!\d)(\d{1,2})\s*[-./]\s*(\d{1,2})(?!\d)",
        normalized,
    ):
        overlaps = any(
            match.start() < end
            and match.end() > start
            for start, end in full_date_spans
        )

        if not overlaps:
            add_candidate(
                rapid_valid_month_day(
                    *match.groups()
                ),
                match,
                "MD_ONLY",
            )

    best_by_date = {}

    for candidate in candidates:
        key = candidate["final_date"]

        if (
            key not in best_by_date
            or candidate["context_score"]
            >
            best_by_date[key]["context_score"]
        ):
            best_by_date[key] = candidate

    return list(
        best_by_date.values()
    )


def rapid_choose_best_candidate(text):
    candidates = rapid_extract_date_candidates(text)

    if not candidates:
        return None

    return max(
        candidates,
        key=lambda candidate: (
            candidate["context_score"],
            candidate["year"] != "NONE",
            -candidate["start"],
        ),
    )


def rapid_items(result):
    return [
        item
        for item in (result or [])
        if isinstance(item, (list, tuple))
        and len(item) >= 3
    ]


IMAGE_INDEX = {
    doctr_normalize_image_id(path.stem): path
    for path in image_paths
}

rapid_required_paths = [
    RAPID_WEIGHTS_DIR / "ch_PP-OCRv4_det_infer.onnx",
    RAPID_WEIGHTS_DIR / "ch_PP-OCRv4_rec_infer.onnx",
    RAPID_WEIGHTS_DIR / "ch_ppocr_mobile_v2.0_cls_infer.onnx",
]

rapid_missing = [str(path) for path in rapid_required_paths if not path.is_file()]
if rapid_missing:
    raise FileNotFoundError(
        "Required offline RapidOCR model files are missing:\n" + "\n".join(rapid_missing)
    )

rapid_ocr = RapidOCR(
    det_model_path=str(RAPID_WEIGHTS_DIR / "ch_PP-OCRv4_det_infer.onnx"),
    rec_model_path=str(RAPID_WEIGHTS_DIR / "ch_PP-OCRv4_rec_infer.onnx"),
    cls_model_path=str(RAPID_WEIGHTS_DIR / "ch_ppocr_mobile_v2.0_cls_infer.onnx"),
    use_gpu=False,
    det_use_cuda=False,
    rec_use_cuda=False,
    cls_use_cuda=False,
)

for image_id in low_confidence_docTR:
    key = doctr_normalize_image_id(image_id)
    image_path = IMAGE_INDEX.get(key)

    if image_path is None:
        continue

    try:
        with Image.open(image_path) as source:
            image = ImageOps.exif_transpose(
                source
            ).convert("RGB")

            image.thumbnail(
                (RAPIDOCR_RESOLUTION, RAPIDOCR_RESOLUTION),
                Image.Resampling.LANCZOS
            )

            image_np = np.ascontiguousarray(
                np.asarray(image)[:, :, ::-1]
            )

        result, _ = rapid_ocr(image_np)
        items = rapid_items(result)

        full_text = " | ".join(
            str(item[1])
            for item in items
        )

        best = rapid_choose_best_candidate(
            full_text
        )

        if best is None:
            continue

        mask = (
            final_results["image_id"].astype(str)
            ==
            str(image_path.stem)
        )

        if not mask.any():
            continue

        final_results.loc[mask, "year"] = best["year"]
        final_results.loc[mask, "month"] = best["month"]
        final_results.loc[mask, "day"] = best["day"]
        final_results.loc[mask, "final_date"] = best["final_date"]

    except Exception:
        continue


FINAL_COLUMNS = [
    "image_id",
    "year",
    "month",
    "day",
    "final_date",
]

df = final_results[FINAL_COLUMNS].copy()

for col in ["image_id", "year", "month", "day"]:
    df[col] = (
        df[col]
        .astype("string")
        .fillna("NONE")
        .replace("", "NONE")
        .astype(str)
    )

mask = df["year"] != "NONE"
df.loc[mask, "year"] = df.loc[mask, "year"].str.zfill(4)

mask = df["month"] != "NONE"
df.loc[mask, "month"] = df.loc[mask, "month"].str.zfill(2)

mask = df["day"] != "NONE"
df.loc[mask, "day"] = df.loc[mask, "day"].str.zfill(2)

all_none = (
    (df["year"] == "NONE")
    & (df["month"] == "NONE")
    & (df["day"] == "NONE")
)

df["final_date"] = (
    df["year"]
    + "-"
    + df["month"]
    + "-"
    + df["day"]
)

df.loc[all_none, "final_date"] = "NONE"

df.to_csv(OUTPUT_PATH, index=False)

print(
    f"Saved {len(df)} rows to {OUTPUT_PATH}"
)
